# Week 6: 신경망 기반 최적화 / Neural-network optimization

오픈소스 HiGHS로 실행합니다. / All solves use open-source HiGHS.

이 notebook, 스크립트와 두 계수 JSON 파일을 같은 폴더에 두고 requirements.txt를 설치합니다. / Keep the notebook, script, and two coefficient JSON files together; install requirements.txt.

In [ ]:
from pathlib import Path
import json
from week06_lab import (make_model, make_omlt, solve_report,
    interval_bounds, tighten_bounds, toy_case, forward)
net = json.loads(Path("frozen_cstr_network.json").read_text())

## 1. 두 뉴런 예제 / Two-neuron example
LP 값 0과 0.4, MILP 값 0.7을 손계산과 비교합니다. / Compare LP values 0 and 0.4 and MILP value 0.7 with hand calculations.

In [ ]:
for tight in [False, True]:
    for relax in [True, False]:
        r = toy_case(tight, relax)
        print(tight, relax, r["objective"], r["termination"])

## 2. Bounds와 MILP / Bounds and MILP
같은 모델에서 구간 전파와 LP-OBBT를 비교합니다. / Compare interval propagation and LP-OBBT on the same model.

In [ ]:
bounds = interval_bounds(net)
tight, preprocessing = tighten_bounds(net, bounds)
for name, limits in [("interval", bounds), ("LP-OBBT", tight)]:
    lp = solve_report(make_model(net, limits, relax=True))
    mip = solve_report(make_model(net, limits))
    print(name, lp["objective"], mip["objective"], mip["binary_variables"])
print("OBBT:", preprocessing)

## 3. OMLT와 forward 검사 / OMLT and forward checks

In [ ]:
result = solve_report(make_omlt(net))
if result["has_incumbent"]:
    physical = [result["temperature"], result["tau"]]
    print("OMLT:", result["outputs"])
    print("Forward:", forward(physical, net))

## 4. 종료 상태 / Termination
가능해가 없으면 입력과 gap을 보고하지 않습니다. / Do not report selected inputs or a gap without an incumbent.

In [ ]:
print(solve_report(make_model(net), limit=0.))
print(solve_report(make_model(net, b_min=2.)))

## 5. 전체 보고서 저장 / Save the full report
제공된 고정 신경망으로 실행합니다. / Run with the supplied frozen network.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "week06_lab.py", "--output-dir", "my-results"], check=True)
report = json.loads(Path("my-results/results.json").read_text())
report["comparisons"]